# us smoke days
noaa hms analyst smoke polygons on a 0.5 degree grid over the contiguous us; smoke days per year at an average location (figure 1a)

In [1]:
from pathlib import Path
import io, zipfile
import numpy as np, pandas as pd, shapefile, shapely
from shapely.geometry import shape

raw, data = Path("../data/raw"), Path("data")


def polygons(f):
    with zipfile.ZipFile(f) as z:
        shp = next(n for n in z.namelist() if n.endswith(".shp"))
        rd = shapefile.Reader(shp=io.BytesIO(z.read(shp)), dbf=io.BytesIO(z.read(shp[:-4] + ".dbf")),
                              shx=io.BytesIO(z.read(shp[:-4] + ".shx")))
        return [shape(r.shape.__geo_interface__) for r in rd.iterShapeRecords()]


# grid cell centres inside the census national boundary
us = shapely.union_all(polygons(raw / "boundaries" / "cb_2023_us_nation_20m.zip"))
lon, lat = np.meshgrid(np.arange(-124.75, -66.5, 0.5), np.arange(24.75, 49.5, 0.5))
keep = shapely.contains_xy(us, lon.ravel(), lat.ravel())
lon, lat = lon.ravel()[keep], lat.ravel()[keep]

In [2]:
days = {}
for f in sorted((raw / "hms").glob("hms_smoke*.zip")):
    y = int(f.stem[9:13])
    try:
        polys = polygons(f)
    except Exception:
        continue  # unreadable archive file
    hit = np.zeros(len(lon), bool)
    for g in polys:
        g = g if g.is_valid else g.buffer(0)
        x0, y0, x1, y1 = g.bounds
        box = (lon >= x0) & (lon <= x1) & (lat >= y0) & (lat <= y1)
        if box.any():
            hit[box] |= shapely.contains_xy(g, lon[box], lat[box])
    days[y] = days.get(y, 0) + hit.astype(int)
ann = pd.DataFrame({"year": list(days), "smoke_days": [v.mean() for v in days.values()]})
ann[ann.year > 2005].to_csv(data / "hms_smoke_days.csv", index=False)
ann.round(1)

Out[2]: 
    year  smoke_days
0   2005         3.6
1   2006        16.8
2   2007        26.0
3   2008        24.0
4   2009        16.0
5   2010        16.6
6   2011        32.4
7   2012        51.2
8   2013        37.0
9   2014        23.7
10  2015        32.5
11  2016        25.3
12  2017        41.4
13  2018        42.9
14  2019        33.0
15  2020        50.7
16  2021        88.6
17  2022       104.9
18  2023       133.3
19  2024       113.9
20  2025        87.8
21  2026        88.4


,year,smoke_days
0,2005,3.6
1,2006,16.8
2,2007,26.0
3,2008,24.0
4,2009,16.0
5,2010,16.6
6,2011,32.4
7,2012,51.2
8,2013,37.0
9,2014,23.7
